# MISSION 1(Fine-Tuning)


In [1]:
# 사전학습 학습 모델 불러오기
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [3]:
# 샘플 tokenize 해보기
sample_text = "이 영화 정말 재밌다"

encoded = tokenizer(sample_text)

print(encoded)

{'input_ids': [2, 1504, 3771, 3944, 7478, 2062, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [4]:
# Fine-tuned 모델 불러오기
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# 모델 화긴해보기
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

# MISSION 2(RAG)


In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [7]:
# 샘플 지식베이스

knowledge_base = [
    "이 영화는 2023년 개봉한 한국 액션 블록버스터로 관객 800만 명을 돌파했다.",
    "주연 배우의 연기력이 뛰어나다는 평이 많으며 CG 퀄리티가 훌륭하다.",
    "스토리가 진부하고 예측 가능하다는 부정적인 의견도 존재한다.",
    "OST가 매우 감동적이며 엔딩 크레딧까지 자리를 뜨지 못하게 만든다.",
    "러닝타임이 2시간 30분으로 다소 길지만 지루하지 않다는 반응이다.",
    "아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다.",
]

In [8]:
# 지식베이스 문장을 벡터로 변환.
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

kb_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
# 벡터 검색 인덱싱(빠르게 찾자!!)
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)

In [10]:
# 질문과 비슷한 문서를 검색하는 함수
def retrieve(query, top_k=2):
    query_vec = embedder.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_vec, top_k)

    return [knowledge_base[i] for i in indices[0]]

In [ ]:
# RAG 전체 파이프라인
def rag_sentiment_pipeline(query):
    print(f"[질문]: {query}")

    retrieved = retrieve(query)
    context = " ".join(retrieved)
    print(f"[검색된 문서]: {context}")

    augmented_input = f"{context} {query}"

    # 함수들은 아직 미구현(의도인지 모르겠으나 누락)
    # result = fine_tuned_pipeline(augmented_input[0])

    # print(f"[감정 예측]: {format_result(result)}\n")
    # print("-" * 60)

In [ ]:
# RAG 답변 받아보기
queries = [
    "이 영화 배우 연기가 어때요?",
    "스토리는 어떤가요?",
    "아이맥스로 볼 만한가요?",
]

for q in queries:
    rag_sentiment_pipeline(q)